# RUN in TPU

 # Part 2: RL Oracle Bootstrap, Training, and Validation (GPU-Accelerated)

 This notebook reloads the pre-computed Feature Cube (`AlphaCache`) generated in Part 1,

 bootstraps the environments, and runs PPO training with periodic metric logging and checkpointing.

In [ ]:
import time

start_time = time.time()

In [ ]:
# CELL 1: Setup, Environment Switch, and Path Configuration
import os
import sys
from pathlib import Path

os.environ["CACHE_LOOKBACK"] = "189"
os.environ["CACHE_START_DATE"] = "1998-01-01"


# ============================================================================
# --- ENVIRONMENT SWITCH ---
# Set to True if running in Google Colab.
# Set to False if running locally in Windows PC VSCode.
# ============================================================================
# 1. Detect environment
RUNNING_IN_COLAB = "google.colab" in sys.modules

if RUNNING_IN_COLAB:
    from google.colab import drive

    drive.mount("/content/drive")
    project_path = Path(
        "/content/drive/Othercomputers/My Computer/Files_win10/python/py311/stocks"
    )
    codebase_path = project_path / "notebooks_RLVR_v2"
else:
    project_path = Path("C:/Users/ping/Files_win10/python/py311/stocks")
    codebase_path = project_path / "notebooks_RLVR_v2"

processed_path = project_path / "data" / "processed"
processed_path.mkdir(parents=True, exist_ok=True)

output_path = codebase_path / "output"
output_path.mkdir(parents=True, exist_ok=True)

os.chdir(codebase_path)

if str(codebase_path) not in sys.path:
    sys.path.append(str(codebase_path))

# Verify environment visibility
print("\n--- Environment Check ---")
print(f"Current Directory: {os.getcwd()}")
visible_files = os.listdir()
print(f"Files visible here: {visible_files}")

if "core" in visible_files and "rl_discovery" in visible_files:
    print("\n[OK] SUCCESS: Python can see your architecture! Imports will now work.")
    print(f"project_path: {project_path}")
    print(f"processed_path: {processed_path}")
    print(f"codebase_path: {codebase_path}")
    print(f"output_path: {output_path}")
else:
    print("\n[ERROR] FAILURE: Python cannot see 'core' or 'rl_discovery'.")
    print("Please check your directory configuration and folder structure.")

In [ ]:
# CELL 3: Imports
import torch
import numpy as np
import plotly.graph_objects as go
import pickle
from typing import cast

# --- Domain Imports ---
from core.logic import AlphaLogic
from core.settings import TradingConfig, CacheConfig
from data_pipeline.screener import UniverseScreener
from rl_discovery.oracle import RLOracle
from rl_discovery.environment import DiscoveryEnv
from rl_discovery.adapter import RLVRGymEnv
from rl_discovery.agent import AbsoluteZeroAgent
from rl_discovery.trainer import RolloutBuffer, PPOTrainer
from rl_discovery.validator import AgentEvaluator

# TODO retrieve them to `processed_path = project_path / "data/processed"`

In [ ]:
import os
import pandas as pd

print(f"Retrieving dataframes from '{processed_path}'...")

# 1. Retrieve df_ohlcv
ohlcv_path = os.path.join(processed_path, "df_ohlcv.parquet")
if os.path.exists(ohlcv_path):
    df_ohlcv = pd.read_parquet(ohlcv_path)
    print(f"Successfully loaded df_ohlcv: {len(df_ohlcv)} rows.")
else:
    print(f"Warning: {ohlcv_path} not found!")
    df_ohlcv = None

# 2. Retrieve df_fed
fed_path = os.path.join(processed_path, "df_fed.parquet")
if os.path.exists(fed_path):
    df_fed = pd.read_parquet(fed_path)
    print(f"Successfully loaded df_fed: {len(df_fed)} rows.")
else:
    df_fed = None
    print(f"Note: {fed_path} not found. (df_fed set to None)")

# 3. Retrieve df_indices
indices_path = os.path.join(processed_path, "df_indices.parquet")
if os.path.exists(indices_path):
    df_indices = pd.read_parquet(indices_path)
    print(f"Successfully loaded df_indices: {len(df_indices)} rows.")
else:
    df_indices = None
    print(f"Note: {indices_path} not found. (df_indices set to None)")

# 4 Retrieve macro_df
macro_path = os.path.join(processed_path, "macro_df.parquet")
if os.path.exists(macro_path):
    macro_df = pd.read_parquet(macro_path)
    print(f"Successfully loaded macro_df: {len(macro_df)} rows.")
else:
    macro_df = None
    print(f"Note: {macro_path} not found. (macro_df set to None)")

# 5 Retrieve features_df
features_path = os.path.join(processed_path, "features_df.parquet")
if os.path.exists(features_path):
    features_df = pd.read_parquet(features_path)
    print(f"Successfully loaded features_df: {len(features_df)} rows.")
else:
    features_df = None
    print(f"Note: {features_path} not found. (features_df set to None)")


print("Data retrieval complete.")

In [ ]:
# CELL 5: Reload AlphaCache & Bootstrap Purged Environments
print(
    "\n--- [TASK 2] Reloading AlphaCache and Instantiating Purged V6 Environments ---"
)

# 2. THE CRITICAL FIX: Forcefully sort the unstacked dataframe chronologically
df_close = df_ohlcv["Adj Close"].unstack(level=0).sort_index()
trading_calendar = pd.DatetimeIndex(df_close.dropna(how="all").index.sort_values())

screener = UniverseScreener(
    df_close=df_close,
    features_df=features_df,
    macro_df=macro_df,
    trading_calendar=trading_calendar,
    config=config,
)

cache_file_path = output_path / CacheConfig.get_filename()
feature_cube = pd.read_parquet(cache_file_path)
print(f"feature_cube path:\n{cache_file_path}\n")

oracle = RLOracle(screener, config)
oracle.precompute_reward_matrix(holding_period=config.holding_period)

# 1. STRICT CHRONOLOGICAL PURGED SPLITS
valid_cal = trading_calendar[
    trading_calendar.isin(feature_cube.index.get_level_values("Date").unique())
]

cal_train = valid_cal[
    (valid_cal >= pd.Timestamp("1998-01-01"))
    & (valid_cal <= pd.Timestamp("2017-12-31"))
]
cal_val = valid_cal[
    (valid_cal >= pd.Timestamp("2018-01-16"))
    & (valid_cal <= pd.Timestamp("2022-12-31"))
]
cal_test = valid_cal[(valid_cal >= pd.Timestamp("2023-01-16"))]

# 2. Inject the V6 Environment
env_train = DiscoveryEnv(
    feature_cube, oracle.reward_matrix, cal_train, config.holding_period
)
gym_train = RLVRGymEnv(env_train, macro_df)

env_val = DiscoveryEnv(
    feature_cube, oracle.reward_matrix, cal_val, config.holding_period
)
gym_val = RLVRGymEnv(env_val, macro_df)

env_test = DiscoveryEnv(
    feature_cube, oracle.reward_matrix, cal_test, config.holding_period
)
gym_test = RLVRGymEnv(env_test, macro_df)

print(
    f"[OK] Train Split: {len(cal_train)} days ({cal_train[0].date()} to {cal_train[-1].date()})"
)
print(
    f"[OK] Val Split  : {len(cal_val)} days ({cal_val[0].date()} to {cal_val[-1].date()})"
)
print(
    f"[OK] Test Split : {len(cal_test)} days ({cal_test[0].date()} to {cal_test[-1].date()})"
)

In [ ]:
# CELL 6: PPO Meta-Loop (Frozen Entropy Fixes & Best Model Capture)
import random

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")


def set_seed(seed):
    torch.manual_seed(seed)
    np.random.seed(seed)
    random.seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


# --- THE V6 HYPERPARAMETERS ---
SEEDS = [42, 12345, 987654, 555666777, 20240620, 314159265]
# 1 epoch = 1024 steps (shuffled out of chronological order) = 8 mini_batch
# 1 gradient update after 1 mini_batch
NUM_EPOCHS = 50
NUM_STEPS = 1024
MINI_BATCH_SIZE = 128  # Larger batches for smooth gradients
ENT_COEF = 0.001  # Allows the agent to reduce variance/uncertainty
EVAL_FREQ = 2  # Check Validation every 2 epochs
PATIENCE = 5  # Give it room to breathe

metadata_payload = {
    "SEEDS": SEEDS,
    "NUM_EPOCHS": NUM_EPOCHS,
    "NUM_STEPS": NUM_STEPS,
    "MINI_BATCH_SIZE": MINI_BATCH_SIZE,
    "ENT_COEF": ENT_COEF,
    "EVAL_FREQ": EVAL_FREQ,
    "PATIENCE": PATIENCE,
    "RANK_MAX_OFFSET": config.rank_max_width,
    "HOLDING_PERIOD": config.holding_period,
}

checkpoint_dir = output_path / "model_checkpoints"
checkpoint_dir.mkdir(parents=True, exist_ok=True)

global_best_sharpe = -np.inf
global_best_model_path = None
global_best_seed = None

print("\n--- [TASK 3] Executing PPO V6 Meta-Loop ---")

for seed in SEEDS:
    print(f"\n{'='*50}\n🚀 STARTING TRAINING SEED: {seed}\n{'='*50}")
    set_seed(seed)

    agent = AbsoluteZeroAgent(obs_dim=33, action_dim=13).to(device)
    trainer = PPOTrainer(agent, lr=3e-4, clip_coef=0.2, ent_coef=ENT_COEF)
    buffer = RolloutBuffer(
        num_steps=NUM_STEPS, obs_dim=33, action_dim=13, device=device
    )

    best_val_sharpe = -np.inf
    patience_counter = 0

    for epoch in range(NUM_EPOCHS):
        obs, _ = gym_train.reset()
        epoch_rewards = []

        for step in range(NUM_STEPS):
            obs_tensor = torch.tensor(obs, dtype=torch.float32).to(device)
            with torch.no_grad():
                action, logprob, _, value = agent.get_action_and_value(
                    obs_tensor.unsqueeze(0)
                )

            raw_action = action.cpu().numpy()[0]
            next_obs, reward, done, _, info = gym_train.step(raw_action)

            buffer.add(obs, action[0], logprob[0], reward, value[0], done)
            epoch_rewards.append(reward)

            obs = next_obs
            if done:
                obs, _ = gym_train.reset()

        obs_tensor = torch.tensor(obs, dtype=torch.float32).unsqueeze(0).to(device)
        with torch.no_grad():
            next_value = agent.get_value(obs_tensor).flatten()

        buffer.compute_advantages(next_value, next_done=done)
        diagnostics = trainer.update(
            buffer, update_epochs=4, mini_batch_size=MINI_BATCH_SIZE
        )
        buffer.step = 0

        if (epoch + 1) % EVAL_FREQ == 0:
            print(
                f"Epoch {epoch+1:03d} | Rew: {np.mean(epoch_rewards):.4f} | Loss: {diagnostics['total_loss']:.4f} | Ent: {diagnostics['entropy']:.4f}"
            )

            val_results = AgentEvaluator.evaluate(agent, gym_val, device)
            val_sharpe = val_results["sharpe_ratio"]
            val_ret = val_results["total_return"] * 100

            print(f"   -> [VAL] Sharpe: {val_sharpe:.3f} | Return: {val_ret:.2f}%")

            if val_sharpe > best_val_sharpe:
                best_val_sharpe = val_sharpe
                patience_counter = 0

                # Dynamic Filename
                dynamic_name = (
                    f"champion_seed_{seed}_ep_{epoch+1}_sh_{val_sharpe:.2f}.pt"
                )
                seed_best_path = checkpoint_dir / dynamic_name
                torch.save(agent.state_dict(), str(seed_best_path))
                print(f"   -> 🟢 NEW BEST! Saved as: {dynamic_name}")

                # Track Global
                if val_sharpe > global_best_sharpe:
                    global_best_sharpe = val_sharpe
                    global_best_model_path = seed_best_path
                    global_best_seed = seed
            else:
                patience_counter += 1
                print(
                    f"   -> 🔴 No improvement. Patience: {patience_counter}/{PATIENCE}"
                )

            if patience_counter >= PATIENCE:
                print(f"🛑 EARLY STOPPING triggered for Seed {seed}")
                break

print(f"\n{'*'*50}")
print(f"🏆 GLOBAL CHAMPION CROWNED 🏆")
print(f"Best Seed:   {global_best_seed}")
print(f"Best Sharpe: {global_best_sharpe:.3f}")
print(f"File Path:   {global_best_model_path.name}")
print(f"{'*'*50}\n")

In [ ]:
# CELL 7: OOS Deterministic Evaluation & Blotter Review
import json

print("\n--- [TASK 4] Walk-Forward Deterministic Evaluation (OOS) ---")

# 1. Load the Ultimate Champion Model
champion_agent = AbsoluteZeroAgent(obs_dim=33, action_dim=13).to(device)
champion_agent.load_state_dict(torch.load(global_best_model_path))
print(f"[LOADED] Champion weights restored from: {global_best_model_path.name}")

# 2. Build Metadata Payload
metadata_payload["BEST_SEED"] = global_best_seed
metadata_payload["CHAMPION_FILE"] = global_best_model_path.name

# 3. Evaluate using the V6 Glass Box
results = AgentEvaluator.evaluate(
    champion_agent, gym_test, device, detailed_log=True, metadata=metadata_payload
)

print(f"\n[OOS TEST METRICS]")
print(f"Total Cumulative Return: {results['total_return']*100:.2f}%")
print(f"Sharpe Ratio (Annualized): {results['sharpe_ratio']:.3f}")
print(f"Total Trading Steps: {results['steps']}")

# 4. Save results
import pickle

results_save_path = output_path / "oos_evaluation_results_v6.pkl"
with open(results_save_path, "wb") as f:
    pickle.dump(results, f)
print(f"\n[SAVED] Complete V6 evaluation dictionary saved to {results_save_path}")

# 5. Print the Glass Box Blotter (First 2 days)
print("\n--- V6 GLASS BOX BLOTTER INSPECTION (First 2 Steps) ---")
blotter = results["blotter"]
for i in range(2):
    print(json.dumps(blotter[i], indent=2))

In [ ]:
end_time = time.time()
total_time = end_time - start_time

# Calculate hours, minutes, and seconds
hours, remainder = divmod(int(total_time), 3600)
minutes, seconds = divmod(remainder, 60)

print(f"\nTotal notebook execution time: {hours:02d}:{minutes:02d}:{seconds:02d}")